## Import Libraries

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
# import os

In [2]:
# Download required nltk resources
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Load Data

In [3]:
# Load data
train_df = pd.read_csv('C:/Users/Acer/Desktop/my all projects/LLM classification/train.csv')
test_df = pd.read_csv('C:/Users/Acer/Desktop/my all projects/LLM classification/test.csv')
print(train_df.shape)                    
print(test_df.shape)

(57477, 9)
(3, 4)


In [4]:
print(train_df.columns)
print(test_df.columns)

Index(['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b',
       'winner_model_a', 'winner_model_b', 'winner_tie'],
      dtype='object')
Index(['id', 'prompt', 'response_a', 'response_b'], dtype='object')


## Text preprocessing

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [6]:
def preprocess(text):
    # Lowercase and remove special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

In [7]:
# Combine text fields for feature extraction
train_df['combined'] = (
    train_df['prompt'].astype(str) + " " +
    train_df['response_a'].astype(str) + " " +
    train_df['response_b'].astype(str)
)

In [8]:
# Combine text fields for feature extraction
test_df['combined'] = (
    test_df['prompt'].astype(str) + " " +
    test_df['response_a'].astype(str) + " " +
    test_df['response_b'].astype(str)
)

In [9]:
# Apply preprocessing
train_df['combined'] = train_df['combined'].apply(preprocess)
test_df['combined'] = test_df['combined'].apply(preprocess)

In [10]:
# Convert one-hot targets to labels: 0 = A wins, 1 = B wins, 2 = Tie
train_df['label'] = train_df[['winner_model_a', 'winner_model_b', 'winner_tie']].values.argmax(axis=1)

In [11]:
# TF-IDF feature extraction
vectorizer = TfidfVectorizer(max_features=150)
X_train = vectorizer.fit_transform(train_df['combined'])
y_train = train_df['label']

In [12]:
X_test = vectorizer.transform(test_df['combined'])

In [13]:
# Train/validation split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [14]:
# Train Logistic Regression
model = LogisticRegression(solver='saga', max_iter=1000)
model.fit(X_tr, y_tr)

LogisticRegression(max_iter=1000, solver='saga')

In [15]:
# Evaluate
y_val_pred_proba = model.predict_proba(X_val)
print("Validation Log Loss:", log_loss(y_val, y_val_pred_proba))
print("Validation Accuracy:", accuracy_score(y_val, model.predict(X_val)))

Validation Log Loss: 1.0895814201422251
Validation Accuracy: 0.37247738343771747


In [16]:
# Predict test probabilities
y_test_pred_proba = model.predict_proba(X_test)

In [43]:
submission = pd.DataFrame(y_test_pred_proba, columns=['winner_model_a', 'winner_model_b', 'winner_tie'])
submission.insert(0, 'id', test_df['id'])
submission.to_csv("submissionLLM.csv", index=False)